# Portuguese EPUB Extraction

Extract text from the Portuguese edition of *O Livro do Desassossego* by Fernando Pessoa.

**Goal:** Parse the EPUB, extract only numbered fragments (e.g., `[1]`, `[2]`, `[11] Litania`), 
and preserve paragraph structure for later alignment with the English edition.

In [2]:
import ebooklib
from ebooklib import epub
from bs4 import BeautifulSoup
import json
import re

book = epub.read_epub('../data/livro.epub')
print(f"Loaded EPUB with {len(list(book.get_items_of_type(ebooklib.ITEM_DOCUMENT)))} documents")

Loaded EPUB with 529 documents


## Inspect TOC Structure

The EPUB's table of contents uses titles like `[1]`, `[11] Litania`, etc. We need to 
extract the number from these titles to match them with the English edition.

In [3]:
toc = book.toc
all_links = []
for item in toc:
    if isinstance(item, tuple):
        section, links = item
        all_links.extend(links)
    elif hasattr(item, 'href'):
        all_links.append(item)

print(f"Total TOC links: {len(all_links)}")
print("\nFirst 15 links:")
for i, link in enumerate(all_links[:15]):
    print(f"  {i}: '{link.title}' -> {link.href}")

Total TOC links: 556

First 15 links:
  0: 'Coleção Clássicos da Língua Portuguesa - Montecristo Editora Volume 8' -> text/part0001_split_000.html
  1: 'Prefácio' -> text/part0003.html
  2: '[1]' -> text/part0004_split_000.html#heading_id_4
  3: '[2]' -> text/part0004_split_001.html
  4: '[3]' -> text/part0004_split_002.html
  5: '[4]' -> text/part0004_split_003.html
  6: '[5]' -> text/part0004_split_004.html
  7: '[6]' -> text/part0004_split_005.html
  8: '[7]' -> text/part0004_split_006.html
  9: '[8]' -> text/part0004_split_007.html
  10: '[9]' -> text/part0004_split_008.html
  11: '[10]' -> text/part0004_split_009.html
  12: '[11] Litania' -> text/part0004_split_010.html
  13: '[12]' -> text/part0004_split_011.html
  14: '[13]' -> text/part0004_split_012.html


## Extract Numbered Fragments

Filter TOC to only include entries where the title starts with `[number]`. 
Extract paragraph text from each HTML file, preserving `<p>` boundaries.

In [4]:
# Build href to content mapping
href_to_content = {}
for item in book.get_items_of_type(ebooklib.ITEM_DOCUMENT):
    content = item.get_content()
    if isinstance(content, bytes):
        content = content.decode('utf-8', errors='ignore')
    href_to_content[item.get_name()] = content

# Extract numbered fragments
portuguese_sections = {}
seen_hrefs = set()

for link in all_links:
    href = link.href.split('#')[0]
    if href in seen_hrefs or href not in href_to_content:
        continue
    seen_hrefs.add(href)
    
    title = link.title if hasattr(link, 'title') else ''
    match = re.match(r'^\[(\d+)\]', title)
    if not match:
        continue
    
    fragment_num = match.group(1)
    
    soup = BeautifulSoup(href_to_content[href], 'html.parser')
    paragraphs = []
    for p_tag in soup.find_all('p'):
        text = p_tag.get_text().strip()
        text = re.sub(r'\s+', ' ', text)
        if re.match(r'^\[\d+\]$', text):
            continue
        if text and len(text) > 20:
            paragraphs.append(text)
    
    if paragraphs:
        portuguese_sections[fragment_num] = {
            'title': f"Fragment {fragment_num}",
            'paragraphs': paragraphs
        }

print(f"✅ Extracted {len(portuguese_sections)} Portuguese fragments")

✅ Extracted 481 Portuguese fragments


## Save to JSON

In [5]:
with open('../data/fragments.json', 'w', encoding='utf-8') as f:
    json.dump(portuguese_sections, f, ensure_ascii=False, indent=2)

print(f"Saved to data/fragments.json")
print(f"\nSample - Fragment 1 ({len(portuguese_sections['1']['paragraphs'])} paragraphs):")
print(portuguese_sections['1']['paragraphs'][0][:300])

Saved to data/fragments.json

Sample - Fragment 1 (6 paragraphs):
Nasci em um tempo em que a maioria dos jovens haviam perdido a crença em Deus, pela mesma razão que os seus maiores a haviam tido — sem saber porquê. E então, porque o espírito humano tende naturalmente para criticar porque sente, e não porque pensa, a maioria desses jovens escolheu a Humanidade par
